# Download overlapping EMIT and AVIRIS-3 scenes and convolve

End-to-end tutorial using real NASA Earthdata acquisitions over the Angeles National Forest, CA.

**Instruments used**

| | EMIT | AVIRIS-3 |
|---|---|---|
| Platform | ISS (spaceborne) | Airborne |
| Acquisition | 2024-08-25 | 2024-09-05 |
| Bands | 288 good, 366-2500 nm | 284, 390-2494 nm |
| FWHM | 8.4-8.8 nm (nominal) | 7.62-8.31 nm (real file) |
| Collection | EMITL2ARFL v001 @ LP DAAC | AV3_L2A_RFL_2357 v1 @ ORNL DAAC |

We download the AVIRIS-3 granule (~886 MB), convolve it to EMIT spectral sampling
using `srfforge`, and compare the results.

**Note on the EMIT download:** we use the bundled nominal EMIT wavelengths rather than
downloading the full 1.8 GB EMIT product. The per-acquisition band centres differ
negligibly from the nominal values. To use a real EMIT product file instead:
```python
emit = EMIT(srf_file="EMIT_L2A_RFL_001_20240825T173115_2423811_008.nc")
```


---
## 1. Setup

### Install dependencies
```bash
pip install -e ".[tutorials]"   # from the srfforge repo root
```

### NASA Earthdata account
1. Register (free) at https://urs.earthdata.nasa.gov/users/new
2. Run the login cell once — credentials are saved to `~/.netrc` for future sessions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from pathlib import Path

import earthaccess

from srfforge import BandConvolver, EMIT, AVIRIS3
from srfforge.io import read_aviris3_nc


In [ ]:
# persist=True saves credentials to ~/.netrc so you only need to do this once
auth = earthaccess.login(persist=True)
print("Authenticated:", auth.authenticated)


---
## 2. Search region

Bounding box `(west, south, east, north)` over the Angeles National Forest, CA.
Varied land cover (chaparral, conifer forest, urban edges) makes it a good test scene.


In [ ]:
# (west, south, east, north) in decimal degrees
BBOX = (-118.5, 34.0, -117.5, 34.8)

fig, ax = plt.subplots(figsize=(5, 4))
rect = mpatches.Rectangle(
    (BBOX[0], BBOX[1]), BBOX[2] - BBOX[0], BBOX[3] - BBOX[1],
    linewidth=2, edgecolor="steelblue", facecolor="steelblue", alpha=0.3,
)
ax.add_patch(rect)
ax.set_xlim(BBOX[0] - 1, BBOX[2] + 1)
ax.set_ylim(BBOX[1] - 1, BBOX[3] + 1)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Search bounding box")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## 3. Search for granules

Neither product supports on-demand spatial subsetting, so full granules are downloaded.

| Collection | Short name | DAAC |
|---|---|---|
| EMIT L2A Reflectance | EMITL2ARFL v001 | LP DAAC |
| AVIRIS-3 L2A Reflectance | AV3_L2A_RFL_2357 v1 | ORNL DAAC |


In [ ]:
emit_results = earthaccess.search_data(
    short_name="EMITL2ARFL", version="001", bounding_box=BBOX, count=100,
)
av3_results = earthaccess.search_data(
    short_name="AV3_L2A_RFL_2357", version="1", bounding_box=BBOX, count=100,
)
print(f"EMIT granules found    : {len(emit_results)}")
print(f"AVIRIS-3 granules found: {len(av3_results)}")


---
## 4. Find overlapping granule pairs

Extract bounding boxes from CMR metadata, find overlapping pairs, rank by overlap area.

All AVIRIS-3 granules here are from one flight day: **2024-09-05**.
The closest EMIT acquisition is **2024-08-25** (11 days prior).


In [ ]:
def get_bbox(g):
    """Return (west, south, east, north) from a CMR DataGranule."""
    geom = (
        g["umm"].get("SpatialExtent", {})
        .get("HorizontalSpatialDomain", {})
        .get("Geometry", {})
    )
    if "BoundingRectangles" in geom:
        bb = geom["BoundingRectangles"][0]
        return (bb["WestBoundingCoordinate"], bb["SouthBoundingCoordinate"],
                bb["EastBoundingCoordinate"], bb["NorthBoundingCoordinate"])
    pts = geom["GPolygons"][0]["Boundary"]["Points"]
    lons = [p["Longitude"] for p in pts]
    lats = [p["Latitude"] for p in pts]
    return (min(lons), min(lats), max(lons), max(lats))

def overlap_area(a, b):
    w = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return w * h

def get_time(g):
    return g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10]

def get_ur(g):
    return g["umm"]["GranuleUR"]


emit_aug25 = [g for g in emit_results if get_time(g) == "2024-08-25"]
emit_combined_bbox = (
    min(get_bbox(g)[0] for g in emit_aug25),
    min(get_bbox(g)[1] for g in emit_aug25),
    max(get_bbox(g)[2] for g in emit_aug25),
    max(get_bbox(g)[3] for g in emit_aug25),
)
print(f"EMIT 2024-08-25 granules : {len(emit_aug25)}")
print(f"Combined EMIT bbox       : {tuple(round(x, 2) for x in emit_combined_bbox)}")
print()

scored = sorted(
    [(overlap_area(emit_combined_bbox, get_bbox(g)), g) for g in av3_results],
    reverse=True,
)
print("Top 5 AVIRIS-3 granules by overlap with EMIT 2024-08-25:")
for area, g in scored[:5]:
    bb = get_bbox(g)
    w_km = max(0, min(bb[2], emit_combined_bbox[2]) - max(bb[0], emit_combined_bbox[0])) * 111
    h_km = max(0, min(bb[3], emit_combined_bbox[3]) - max(bb[1], emit_combined_bbox[1])) * 111
    print(f"  {get_ur(g)}  overlap ~{w_km:.1f} x {h_km:.1f} km")


In [ ]:
# Chosen granule: best overlap with EMIT 2024-08-25 AND smallest file size (886 MB)
# Overlap with EMIT_L2A_RFL_001_20240825T173115_2423811_008: ~6.3 x 5.0 km
chosen_av3 = next(g for g in av3_results
                  if get_ur(g) == "AV320240905t182728_000_L2A_RFL_1")

print(f"Granule : {get_ur(chosen_av3)}")
print(f"Date    : {get_time(chosen_av3)}")
print(f"Bbox    : {tuple(round(x, 2) for x in get_bbox(chosen_av3))}")
print()
print("Files in this granule:")
for link in chosen_av3.data_links():
    print(f"  {link.split(chr(47))[-1]}")


---
## 5. Download

File inventory for this granule:
- `_RFL_ORT.nc` — orthorectified surface reflectance (886 MB) **we need this**
- `_UNC_ORT.nc` — per-pixel uncertainty (872 MB)
- `_RFL_ORT_QL.tif` — RGB quicklook GeoTIFF (4 MB)
- `.yaml` — processing metadata

`earthaccess.download` fetches all files in the granule.
We skip the EMIT file (1.8 GB) and use the bundled nominal wavelengths.


In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

av3_files = earthaccess.download([chosen_av3], local_path=data_dir)

print("Downloaded:")
for f in av3_files:
    size_mb = Path(f).stat().st_size / 1e6
    print(f"  {Path(f).name}  ({size_mb:.0f} MB)")


---
## 6. Load the AVIRIS-3 file

`read_aviris3_nc` reads the `reflectance` group and transposes from
bands-first `(284, 1532, 1605)` to `(1532, 1605, 284)` so the band axis is last.

Confirmed file structure:
```
/reflectance/
    reflectance   (284, 1532, 1605)  float32   bands x lines x cols
    wavelength    (284,)             float32   nm, 389.8 to 2493.5
    fwhm          (284,)             float32   nm, 7.62 to 8.31
/easting          (1605,)            float64
/northing         (1532,)            float64
/aerosol_optical_thickness/...
/water_vapor/...
```


In [ ]:
av3_nc = next(f for f in av3_files if f.endswith("_RFL_ORT.nc"))

av3_data, av3, av3_meta = read_aviris3_nc(av3_nc)

# Replace fill value with NaN before any arithmetic
av3_data[av3_data == av3_meta["no_data"]] = np.nan

print(f"Shape      : {av3_data.shape}  (lines, cols, bands)")
print(f"Bands      : {len(av3.wavelengths)}  ({av3.wavelengths[0]:.1f} - {av3.wavelengths[-1]:.1f} nm)")
print(f"FWHM range : {av3.fwhm.min():.2f} - {av3.fwhm.max():.2f} nm")
print(av3)


---
## 7. Convolve to EMIT spectral sampling

`EMIT()` loads the bundled nominal wavelengths (288 bands, 366-2500 nm) from
emit-sds/emit-sds-l1b. `BandConvolver` builds a (288 x 284) Gaussian weight matrix
and applies it as a single matrix multiply over the full 1532 x 1605 scene.


In [ ]:
emit = EMIT()   # bundled nominal wavelengths
print(emit)

conv = BandConvolver(source=av3, target=emit)
print(f"Convolution matrix: {conv.matrix.shape}  (EMIT bands x AVIRIS-3 bands)")

av3_as_emit = conv(av3_data)
print(f"\nAVIRIS-3 original : {av3_data.shape}")
print(f"Convolved to EMIT : {av3_as_emit.shape}")


---
## 8. Visualise results

### 8a. Spatial-mean spectrum (100 x 100 px window near scene centre)

We average over a 100x100 pixel window rather than using a single pixel to reduce
per-pixel noise and give a more representative spectrum of the land cover.


In [ ]:
# 100x100 px mean near scene centre
r0, c0 = av3_data.shape[0] // 2 - 50, av3_data.shape[1] // 2 - 50
mean_av3  = np.nanmean(av3_data[r0:r0+100, c0:c0+100, :],       axis=(0, 1))
mean_emit = np.nanmean(av3_as_emit[r0:r0+100, c0:c0+100, :],    axis=(0, 1))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(av3.wavelengths,  mean_av3,  lw=1,   color="darkorange", alpha=0.8,
        label=f"AVIRIS-3  ({len(av3.wavelengths)} bands, {av3.fwhm.mean():.1f} nm FWHM)")
ax.plot(emit.wavelengths, mean_emit, lw=1.8, color="steelblue",
        label=f"Convolved to EMIT  ({len(emit.wavelengths)} bands, {emit.fwhm.mean():.1f} nm FWHM)")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance")
ax.set_title(
    "Spatial mean (100x100 px): AVIRIS-3 vs convolved to EMIT\n"
    "2024-09-05, Angeles NF, CA  |  AV320240905t182728_000"
)
ax.legend()
ax.set_xlim(350, 2550)
ax.set_ylim(0, None)
ax.xaxis.set_minor_locator(ticker.MultipleLocator(100))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 8b. RGB quicklooks


In [ ]:
def make_rgb(d, wl, r_nm=650, g_nm=550, b_nm=450):
    def nearest(w): return int(np.argmin(np.abs(wl - w)))
    rgb = np.stack([d[:, :, nearest(r_nm)],
                    d[:, :, nearest(g_nm)],
                    d[:, :, nearest(b_nm)]], axis=-1)
    p2, p98 = np.nanpercentile(rgb, 2), np.nanpercentile(rgb, 98)
    return np.clip((rgb - p2) / (p98 - p2 + 1e-9), 0, 1)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(make_rgb(av3_data, av3.wavelengths))
axes[0].set_title(f"AVIRIS-3  ({av3_data.shape[2]} bands, {av3.fwhm.mean():.1f} nm FWHM)")
axes[0].axis("off")
axes[1].imshow(make_rgb(av3_as_emit, emit.wavelengths))
axes[1].set_title(f"Convolved to EMIT  ({av3_as_emit.shape[2]} bands, {emit.fwhm.mean():.1f} nm FWHM)")
axes[1].axis("off")
plt.suptitle(
    "RGB quicklooks  |  R=650 nm, G=550 nm, B=450 nm\n"
    "Angeles NF, CA  |  AVIRIS-3 2024-09-05  |  AV320240905t182728_000",
    y=1.01,
)
plt.tight_layout()
plt.show()


### 8c. SRF convolution vs naive linear interpolation

The two methods look similar on a smooth spectrum because AVIRIS-3 and EMIT have
similar spectral resolution (~7-9 nm FWHM). The differences are real but small
(max ~1.3% reflectance units) and concentrate at the **edges** of water vapour
absorption features (~1340 nm and ~1800 nm) where the spectrum changes steeply.
This is exactly where proper SRF convolution matters: steep gradients are weighted
differently by a Gaussian than by linear interpolation.


In [ ]:
mean_interp = np.interp(emit.wavelengths, av3.wavelengths, mean_av3)
residual    = mean_emit - mean_interp

fig = plt.figure(figsize=(13, 6))
gs  = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.4)
ax_main = fig.add_subplot(gs[0, :])
ax_re   = fig.add_subplot(gs[1, 0])
ax_w1   = fig.add_subplot(gs[1, 1])
ax_w2   = fig.add_subplot(gs[1, 2])

# Full residual with highlighted absorption-edge regions
ax_main.plot(emit.wavelengths, residual, color="purple", lw=1)
ax_main.axhline(0, color="k", lw=0.5)
for wl_range, label in [((1300, 1480), "~1400 nm edge"), ((1750, 2020), "~1900 nm edge")]:
    ax_main.axvspan(wl_range[0], wl_range[1], alpha=0.12, color="tomato", label=label)
ax_main.legend(fontsize=8, loc="upper left")
ax_main.set_ylabel("Residual (conv - interp)")
ax_main.set_title(
    f"Full residual  |  max |residual| = {np.nanmax(np.abs(residual)):.4f}\n"
    "Largest differences at water absorption EDGES, not centres"
)
ax_main.set_xlim(350, 2550)
ax_main.grid(True, alpha=0.3)

# Zoomed insets: both curves side-by-side at the steep-gradient regions
for ax, wl_range, title in [
    (ax_re, (670, 770),   "Red edge (670-770 nm)"),
    (ax_w1, (1290, 1480), "Water edge (~1340 nm)"),
    (ax_w2, (1750, 2020), "Water edge (~1800 nm)"),
]:
    mask = (emit.wavelengths >= wl_range[0]) & (emit.wavelengths <= wl_range[1])
    ax.plot(emit.wavelengths[mask], mean_emit[mask],   color="steelblue", lw=1.5, label="SRF conv")
    ax.plot(emit.wavelengths[mask], mean_interp[mask], color="tomato",    lw=1.5, ls="--", label="Interp")
    ax.set_title(title, fontsize=8)
    ax.set_xlabel("nm", fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=7)

plt.suptitle(
    "SRF convolution vs linear interpolation: differences at steep spectral gradients",
    y=1.01,
)
plt.tight_layout()
plt.show()
